In [1]:
%pip install neo4j

Note: you may need to restart the kernel to use updated packages.


In [2]:
from neo4j import GraphDatabase

In [3]:
print("Neo4j Python driver is working!")

Neo4j Python driver is working!


In [6]:
from neo4j import GraphDatabase

URI = "neo4j://127.0.0.1:7687"
USERNAME = "neo4j"
PASSWORD = "AmAwA@2005"

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

driver.verify_connectivity()

print("Successfully connected to Neo4j!")

Successfully connected to Neo4j!


In [7]:
def get_nodes(tx):
    result = tx.run("""
        MATCH (n)
        RETURN elementId(n) AS id, labels(n) AS labels, n
    """)
    return [
        {
            "id": record["id"],
            "labels": record["labels"],
            "properties": dict(record["n"])
        }
        for record in result
    ]

with driver.session(database="neo4j") as session:
    nodes = session.execute_read(get_nodes)

print("Number of nodes:", len(nodes))

Number of nodes: 20


In [8]:
for node in nodes:
    print(node)

{'id': '4:569f1ed3-7b6d-425b-8bff-9fd8b251ee82:0', 'labels': ['Person'], 'properties': {'name': 'Tom Hanks'}}
{'id': '4:569f1ed3-7b6d-425b-8bff-9fd8b251ee82:1', 'labels': ['Person'], 'properties': {'name': 'Robin Wright'}}
{'id': '4:569f1ed3-7b6d-425b-8bff-9fd8b251ee82:2', 'labels': ['Person'], 'properties': {'name': 'Gary Sinise'}}
{'id': '4:569f1ed3-7b6d-425b-8bff-9fd8b251ee82:3', 'labels': ['Person'], 'properties': {'name': 'Robert Zemeckis'}}
{'id': '4:569f1ed3-7b6d-425b-8bff-9fd8b251ee82:4', 'labels': ['Person'], 'properties': {'name': 'Tim Robbins'}}
{'id': '4:569f1ed3-7b6d-425b-8bff-9fd8b251ee82:5', 'labels': ['Person'], 'properties': {'name': 'Leonardo DiCaprio'}}
{'id': '4:569f1ed3-7b6d-425b-8bff-9fd8b251ee82:6', 'labels': ['Person'], 'properties': {'name': 'Kate Winslet'}}
{'id': '4:569f1ed3-7b6d-425b-8bff-9fd8b251ee82:7', 'labels': ['Person'], 'properties': {'name': 'Christopher Nolan'}}
{'id': '4:569f1ed3-7b6d-425b-8bff-9fd8b251ee82:8', 'labels': ['Person'], 'properties': {

In [9]:
def get_relationships(tx):
    result = tx.run("""
        MATCH (n)-[r]->(m)
        RETURN elementId(n) AS source,
               type(r) AS relationship_type,
               elementId(m) AS target
    """)
    
    return [
        {
            "source": record["source"],
            "relationship_type": record["relationship_type"],
            "target": record["target"]
        }
        for record in result
    ]

with driver.session(database="neo4j") as session:
    relationships = session.execute_read(get_relationships)

print("Number of relationships:", len(relationships))

Number of relationships: 22


In [10]:
%pip install bokeh networkx pandas

Note: you may need to restart the kernel to use updated packages.


In [11]:
import bokeh
import networkx
import pandas

print("Bokeh version:", bokeh.__version__)
print("NetworkX version:", networkx.__version__)
print("Pandas version:", pandas.__version__)

Bokeh version: 3.9.2
NetworkX version: 3.6.1
Pandas version: 2.2.3


In [12]:
import networkx as nx
from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.models import HoverTool
from bokeh.palettes import Category10
from bokeh.plotting import from_networkx

In [13]:
G = nx.DiGraph()

# Add nodes
for node in nodes:
    G.add_node(
        node["id"],
        labels=node["labels"],
        properties=node["properties"]
    )

# Add relationships
for rel in relationships:
    G.add_edge(
        rel["source"],
        rel["target"],
        relationship_type=rel["relationship_type"]
    )

print("Nodes in NetworkX:", G.number_of_nodes())
print("Relationships in NetworkX:", G.number_of_edges())

Nodes in NetworkX: 20
Relationships in NetworkX: 22


In [14]:
output_notebook()

plot = figure(
    width=900,
    height=650,
    title="Neo4j Network Visualization",
    x_axis_type=None,
    y_axis_type=None,
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

network_graph = from_networkx(
    G,
    nx.spring_layout,
    scale=2,
    center=(0, 0)
)

plot.renderers.append(network_graph)

show(plot)

Loading BokehJS ...

In [15]:
for node_id, data in G.nodes(data=True):
    if "Person" in data["labels"]:
        G.nodes[node_id]["node_type"] = "Person"
        G.nodes[node_id]["display_name"] = data["properties"]["name"]
    elif "Movie" in data["labels"]:
        G.nodes[node_id]["node_type"] = "Movie"
        G.nodes[node_id]["display_name"] = data["properties"]["title"]

print("Node information added successfully!")

Node information added successfully!


In [16]:
for node_id, data in G.nodes(data=True):
    print(data["node_type"], ":", data["display_name"])

Person : Tom Hanks
Person : Robin Wright
Person : Gary Sinise
Person : Robert Zemeckis
Person : Tim Robbins
Person : Leonardo DiCaprio
Person : Kate Winslet
Person : Christopher Nolan
Person : Matthew McConaughey
Person : Anne Hathaway
Person : Christian Bale
Person : Steven Spielberg
Movie : Forrest Gump
Movie : The Terminal
Movie : Catch Me If You Can
Movie : Titanic
Movie : Inception
Movie : Interstellar
Movie : The Dark Knight
Movie : The Dark Knight Rises


In [17]:
#PHASE 6 — MAKE PERSON AND MOVIE NODES DIFFERENT
#Create a better Bokeh graph
output_notebook()

plot = figure(
    width=1000,
    height=700,
    title="Interactive Movie Network",
    x_axis_type=None,
    y_axis_type=None,
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

network_graph = from_networkx(
    G,
    nx.spring_layout,
    scale=2,
    center=(0, 0)
)

plot.renderers.append(network_graph)

show(plot)

Loading BokehJS ...

In [18]:
#PHASE 7 
from bokeh.io import output_notebook, show
from bokeh.plotting import figure, from_networkx
from bokeh.models import HoverTool

output_notebook()

plot = figure(
    width=1000,
    height=700,
    title="Interactive Movie Network",
    x_axis_type=None,
    y_axis_type=None,
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

network_graph = from_networkx(
    G,
    nx.spring_layout,
    scale=2,
    center=(0, 0)
)

plot.renderers.append(network_graph)

# Add hover tool
hover = HoverTool(
    renderers=[network_graph.node_renderer],
    tooltips=[
        ("Name / Title", "@display_name"),
        ("Type", "@node_type")
    ]
)

plot.add_tools(hover)

show(plot)


Loading BokehJS ...

In [19]:
for node_id, data in G.nodes(data=True):

    if data["node_type"] == "Movie":
        data["year"] = data["properties"]["year"]
        data["genre"] = data["properties"]["genre"]
    else:
        data["year"] = ""
        data["genre"] = ""

print("Movie information prepared!")

Movie information prepared!


In [20]:
output_notebook()

plot = figure(
    width=1000,
    height=700,
    title="Interactive Movie Network",
    x_axis_type=None,
    y_axis_type=None,
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

network_graph = from_networkx(
    G,
    nx.spring_layout,
    scale=2,
    center=(0, 0)
)

plot.renderers.append(network_graph)

hover = HoverTool(
    renderers=[network_graph.node_renderer],
    tooltips=[
        ("Name / Title", "@display_name"),
        ("Type", "@node_type"),
        ("Year", "@year"),
        ("Genre", "@genre")
    ]
)

plot.add_tools(hover)

show(plot)

Loading BokehJS ...

In [21]:
# PHASE 8.6 - Improve the Graph Appearance
from bokeh.models import Circle, Scatter

# Set different sizes for Person and Movie nodes

for node_id, data in G.nodes(data=True):

    if data["node_type"] == "Person":
        G.nodes[node_id]["size"] = 20
        G.nodes[node_id]["marker"] = "circle"

    elif data["node_type"] == "Movie":
        G.nodes[node_id]["size"] = 28
        G.nodes[node_id]["marker"] = "square"

print("Node appearance information added!")


Node appearance information added!


In [22]:
for node_id, data in G.nodes(data=True):
    print(
        data["display_name"],
        "| Type:", data["node_type"],
        "| Size:", data["size"],
        "| Marker:", data["marker"]
    )

Tom Hanks | Type: Person | Size: 20 | Marker: circle
Robin Wright | Type: Person | Size: 20 | Marker: circle
Gary Sinise | Type: Person | Size: 20 | Marker: circle
Robert Zemeckis | Type: Person | Size: 20 | Marker: circle
Tim Robbins | Type: Person | Size: 20 | Marker: circle
Leonardo DiCaprio | Type: Person | Size: 20 | Marker: circle
Kate Winslet | Type: Person | Size: 20 | Marker: circle
Christopher Nolan | Type: Person | Size: 20 | Marker: circle
Matthew McConaughey | Type: Person | Size: 20 | Marker: circle
Anne Hathaway | Type: Person | Size: 20 | Marker: circle
Christian Bale | Type: Person | Size: 20 | Marker: circle
Steven Spielberg | Type: Person | Size: 20 | Marker: circle
Forrest Gump | Type: Movie | Size: 28 | Marker: square
The Terminal | Type: Movie | Size: 28 | Marker: square
Catch Me If You Can | Type: Movie | Size: 28 | Marker: square
Titanic | Type: Movie | Size: 28 | Marker: square
Inception | Type: Movie | Size: 28 | Marker: square
Interstellar | Type: Movie | Siz

In [23]:
# PHASE 9 — Connected-Node Highlighting
# Create a mapping from node ID to node index

node_ids = list(G.nodes())

node_index = {
    node_id: index
    for index, node_id in enumerate(node_ids)
}

print("Number of nodes:", len(node_ids))

Number of nodes: 20


In [24]:
# Create a dictionary containing the connected nodes

connected_nodes = {}

for node_id in G.nodes():

    neighbors = set()

    # Nodes connected from this node
    for target in G.successors(node_id):
        neighbors.add(target)

    # Nodes connected to this node
    for source in G.predecessors(node_id):
        neighbors.add(source)

    connected_nodes[node_id] = neighbors

print("Connection map created successfully!")

Connection map created successfully!


In [25]:
for node_id, neighbors in connected_nodes.items():

    print(
        G.nodes[node_id]["display_name"],
        "->",
        [G.nodes[n]["display_name"] for n in neighbors]
    )

Tom Hanks -> ['Forrest Gump', 'The Terminal']
Robin Wright -> ['Forrest Gump']
Gary Sinise -> ['Forrest Gump']
Robert Zemeckis -> ['Forrest Gump', 'The Terminal']
Tim Robbins -> ['Forrest Gump']
Leonardo DiCaprio -> ['Titanic', 'Inception', 'Catch Me If You Can']
Kate Winslet -> ['Titanic', 'Inception']
Christopher Nolan -> ['The Dark Knight', 'Inception', 'The Dark Knight Rises', 'Interstellar']
Matthew McConaughey -> ['Interstellar']
Anne Hathaway -> ['Interstellar']
Christian Bale -> ['The Dark Knight', 'The Dark Knight Rises']
Steven Spielberg -> ['Titanic', 'Catch Me If You Can']
Forrest Gump -> ['Gary Sinise', 'Robert Zemeckis', 'Robin Wright', 'Tim Robbins', 'Tom Hanks']
The Terminal -> ['Robert Zemeckis', 'Tom Hanks']
Catch Me If You Can -> ['Leonardo DiCaprio', 'Steven Spielberg']
Titanic -> ['Leonardo DiCaprio', 'Kate Winslet', 'Steven Spielberg']
Inception -> ['Leonardo DiCaprio', 'Kate Winslet', 'Christopher Nolan']
Interstellar -> ['Matthew McConaughey', 'Christopher Nolan

In [26]:
from bokeh.models import ColumnDataSource

node_data = {
    "index": list(range(len(node_ids))),
    "node_id": node_ids,
    "display_name": [],
    "node_type": [],
    "year": [],
    "genre": [],
    "alpha": []
}

for node_id in node_ids:

    data = G.nodes[node_id]

    node_data["display_name"].append(data["display_name"])
    node_data["node_type"].append(data["node_type"])
    node_data["year"].append(data["year"])
    node_data["genre"].append(data["genre"])

    # All nodes start fully visible
    node_data["alpha"].append(1.0)

node_source = ColumnDataSource(node_data)

print("Bokeh node data created!")

Bokeh node data created!


In [28]:
# Create the graph
from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.models import GraphRenderer, StaticLayoutProvider
from bokeh.models import Circle
import networkx as nx

output_notebook()

plot = figure(
    width=1000,
    height=700,
    title="Interactive Movie Network - Connected Node Highlighting",
    x_axis_type=None,
    y_axis_type=None,
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

# Create the GraphRenderer
graph = GraphRenderer()

# Add node data
graph.node_renderer.data_source = node_source

# Normal node appearance
graph.node_renderer.glyph = Circle(
    radius=0.08,
    fill_alpha="alpha"
)

# Selected node appearance
graph.node_renderer.selection_glyph = Circle(
    radius=0.12,
    fill_alpha=1.0
)

# Hover node appearance
graph.node_renderer.hover_glyph = Circle(
    radius=0.10,
    fill_alpha=1.0
)

# Create edge information
edge_start = []
edge_end = []
edge_types = []

for source, target, data in G.edges(data=True):

    edge_start.append(node_index[source])
    edge_end.append(node_index[target])
    edge_types.append(data["relationship_type"])

# Add edge data
graph.edge_renderer.data_source.data = {
    "start": edge_start,
    "end": edge_end,
    "relationship_type": edge_types
}

# Edge appearance
graph.edge_renderer.glyph.line_alpha = 0.5
graph.edge_renderer.glyph.line_width = 2

# Create graph positions
positions = nx.spring_layout(
    G,
    seed=42,
    scale=2
)

graph_layout = {}

for node_id, position in positions.items():

    graph_layout[node_index[node_id]] = position

graph.layout_provider = StaticLayoutProvider(
    graph_layout=graph_layout
)

# Add graph to plot
plot.renderers.append(graph)

show(plot)

Loading BokehJS ...

In [29]:
# Add the tooltip
from bokeh.models import HoverTool

hover = HoverTool(
    renderers=[graph.node_renderer],
    tooltips=[
        ("Name / Title", "@display_name"),
        ("Type", "@node_type"),
        ("Year", "@year"),
        ("Genre", "@genre")
    ]
)

plot.add_tools(hover)

print("Hover tooltip added successfully!")

Hover tooltip added successfully!


In [30]:
show(plot)

In [31]:
# Now add the highlighting
from bokeh.models import CustomJS

callback = CustomJS(
    args=dict(
        node_source=node_source,
        graph=graph
    ),
    code="""
    const selected = node_source.selected.indices;

    // If no node is selected
    if (selected.length === 0) {

        for (let i = 0; i < node_source.data.alpha.length; i++) {
            node_source.data.alpha[i] = 1.0;
        }

        node_source.change.emit();
        return;
    }

    // Get the selected node
    const selected_index = selected[0];

    // Store connected nodes
    const connected = new Set();

    const starts = graph.edge_renderer.data_source.data.start;
    const ends = graph.edge_renderer.data_source.data.end;

    // Find connected nodes
    for (let i = 0; i < starts.length; i++) {

        if (starts[i] === selected_index) {
            connected.add(ends[i]);
        }

        if (ends[i] === selected_index) {
            connected.add(starts[i]);
        }
    }

    // Change transparency
    for (let i = 0; i < node_source.data.alpha.length; i++) {

        if (
            i === selected_index ||
            connected.has(i)
        ) {
            node_source.data.alpha[i] = 1.0;
        }
        else {
            node_source.data.alpha[i] = 0.15;
        }
    }

    node_source.change.emit();
    """
)

node_source.selected.js_on_change(
    "indices",
    callback
)

print("Connected-node highlighting enabled!")

Connected-node highlighting enabled!


In [32]:
# PHASE 10 — Search / Filter by Node Attribute
# Import the search widgets
from bokeh.models import TextInput, Button

In [33]:
# Create the search box
search_box = TextInput(
    title="Search Node:",
    placeholder="Type a person or movie name..."
)

print("Search box created!")

Search box created!


In [34]:
#Create the search button
search_button = Button(
    label="Search",
    button_type="primary"
)

print("Search button created!")

Search button created!


In [36]:
# Create the search function
search_callback = CustomJS(
    args=dict(
        node_source=node_source,
        search_box=search_box
    ),
    code="""
    const search_text = search_box.value.toLowerCase().trim();

    // If search box is empty, show all nodes
    if (search_text === "") {

        for (let i = 0; i < node_source.data.alpha.length; i++) {
            node_source.data.alpha[i] = 1.0;
        }

        node_source.change.emit();
        return;
    }

    // Search through node names/titles
    for (let i = 0; i < node_source.data.alpha.length; i++) {

        const name = node_source.data.display_name[i].toLowerCase();

        if (name.includes(search_text)) {
            node_source.data.alpha[i] = 1.0;
        }
        else {
            node_source.data.alpha[i] = 0.15;
        }
    }

    node_source.change.emit();
    """
)

In [37]:
# Connect the Search button
search_button.js_on_click(search_callback)

print("Search button connected!")

Search button connected!


In [38]:
# Create a layout
from bokeh.layouts import column

In [41]:
#Put everything together and Test the search
layout = column(
    search_box,
    search_button,
    plot
)

show(layout)

In [42]:
# PHASE 11 — Relationship-Type Filtering
from bokeh.models import Select

In [43]:
# Create the dropdown
relationship_filter = Select(
    title="Relationship Type:",
    value="ALL",
    options=[
        "ALL",
        "ACTED_IN",
        "DIRECTED"
    ]
)

print("Relationship filter created!")

Relationship filter created!


In [44]:
# Prepare the edge data
graph.edge_renderer.data_source

ColumnDataSource(id='p1344', ...)

In [45]:
edge_source = graph.edge_renderer.data_source

edge_source.data["alpha"] = [
    0.5 for _ in edge_source.data["relationship_type"]
]

print("Edge filter data prepared!")

Edge filter data prepared!


In [47]:
# Create the filtering code
relationship_callback = CustomJS(
    args=dict(
        edge_source=edge_source,
        node_source=node_source,
        relationship_filter=relationship_filter
    ),
    code="""
    const selected_type = relationship_filter.value;

    const relationship_types =
        edge_source.data.relationship_type;

    const alpha =
        edge_source.data.alpha;

    // Show or hide relationships
    for (let i = 0; i < relationship_types.length; i++) {

        if (
            selected_type === "ALL" ||
            relationship_types[i] === selected_type
        ) {
            alpha[i] = 0.5;
        }
        else {
            alpha[i] = 0.0;
        }
    }

    edge_source.change.emit();
    """
)

In [48]:
# Connect the dropdown to the callback
relationship_filter.js_on_change(
    "value",
    relationship_callback
)

print("Relationship filter connected!")

Relationship filter connected!


In [49]:
# Make the edges use alpha
graph.edge_renderer.glyph.line_alpha = "alpha"

print("Edge transparency enabled!")

Edge transparency enabled!


In [50]:
# Add the dropdown to your application
layout = column(
    search_box,
    search_button,
    relationship_filter,
    plot
)

show(layout)

In [51]:
# PHASE 14 — FULL TESTING
with driver.session(database="neo4j") as session:
    nodes = session.execute_read(get_nodes)

print("Number of nodes:", len(nodes))


Number of nodes: 20


In [52]:
with driver.session(database="neo4j") as session:
    relationships = session.execute_read(get_relationships)

print("Number of relationships:", len(relationships))

Number of relationships: 22
